# Challenge 1: Quantum Dice Lab ★☆☆

**The mission:** Build tools powered by true quantum randomness: a coin flipper, dice, a loaded coin, and a fortune teller. Then turn them into something of your own.

**Quantum ideas:** superposition, measurement, probability, rotation angles.
**Best for:** first-time coders and anyone who likes games.
**Time plan:** Missions 1 to 5 take about 75 minutes. Spend the rest of your time on "Make it yours."

**Why quantum?** Python's `random` uses a formula that only *looks* random. If you knew its starting number, you could predict every result. A measured qubit is unpredictable by the laws of physics, which is why real security systems use quantum random number generators.

In [ ]:
# Using Google Colab? Delete the "#" at the start of the next line, then run this cell once.
# !pip install -q "qiskit[visualization]"

In [ ]:
# SETUP: run this cell first (click it, then press Shift+Enter).
# You don't need to read or change anything in this cell.

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import math, random, qiskit

_sampler = StatevectorSampler()

def _needs_measure(qc):
    if qc.num_clbits == 0:
        raise ValueError("Your circuit has no measurements yet. Add qc.measure_all() as the last step of your circuit.")

def run(qc, shots=1000):
    """Run a circuit many times. Returns counts, e.g. {'0': 507, '1': 493}."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_counts()

def run_once(qc):
    """Run a circuit one time. Returns a single result as a string, e.g. '101'."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=1).result()[0].join_data().get_bitstrings()[0]

def run_many(qc, shots):
    """Run a circuit `shots` times. Returns every result in order, e.g. ['0', '1', '1', ...]."""
    _needs_measure(qc)
    return _sampler.run([qc], shots=shots).result()[0].join_data().get_bitstrings()

def probabilities(qc):
    """Exact chance of each result, with no randomness. Measurements at the end are ignored."""
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    return {str(k): round(float(v), 4) for k, v in sv.probabilities_dict().items() if v > 1e-9}

def _close(got, want, tol=0.02):
    return all(abs(got.get(k, 0) - want.get(k, 0)) <= tol for k in set(got) | set(want))

def _report(ok, yes, hint):
    if ok:
        print("\u2705 " + yes)
    else:
        print("\u274c Not yet. Hint: " + hint)

def _check_probs(qc, want, yes, hint):
    try:
        ok = _close(probabilities(qc), want)
    except Exception:
        ok = False
    _report(ok, yes, hint)

# ---- checks for this notebook ----
def check_mission_1(flip_coin):
    try:
        flips = [flip_coin() for _ in range(200)]
        ok = set(flips) == {"Heads", "Tails"} and 60 < flips.count("Heads") < 140
    except Exception:
        ok = False
    _report(ok, "Mission 1 complete: a fair quantum coin flipper.",
            "your coin only ever gives one answer. Put the qubit in superposition with qc.h(0).")

def check_mission_2(qc):
    _check_probs(qc, {format(i, "03b"): 0.125 for i in range(8)},
                 "Mission 2 complete: all 8 sides are equally likely.",
                 "every one of the 3 qubits needs its own H gate. A for loop over range(3) works.")

def check_mission_3(roll_d6):
    try:
        rolls = [roll_d6() for _ in range(600)]
        ok = set(rolls) == {1, 2, 3, 4, 5, 6} and all(rolls.count(s) > 50 for s in range(1, 7))
    except Exception:
        ok = False
    _report(ok, "Mission 3 complete: a fair 6-sided die.",
            "your die can still roll 7 or 8. Only return when number < 6; otherwise let the loop roll again.")

def check_mission_4(qc):
    _check_probs(qc, {"0": 0.75, "1": 0.25}, "Mission 4 complete: a coin loaded 75/25.",
                 "solve sin(theta/2) = sqrt(0.25) for theta. math.asin(x) gives the angle whose sin is x.")

def check_mission_5(fortune):
    try:
        got = {fortune("test question") for _ in range(200)}
        ok = got <= set(answers) and len(got) >= 7
    except Exception:
        ok = False
    _report(ok, "Mission 5 complete: your fortune teller is powered by qubits.",
            "get a number from 0 to 7 with int(run_once(make_d8_circuit()), 2), then return answers[number].")

print(f"\u2705 Setup complete (Qiskit {qiskit.__version__}). You're ready to go!")

## Mission 1: Quantum coin flipper

Wrap a one-qubit coin inside a Python function so you can flip it whenever you want.

**TODO:** Put the qubit into a 50/50 superposition.

In [ ]:
def flip_coin():
    qc = QuantumCircuit(1)
    # TODO: put the qubit into a 50/50 superposition

    qc.measure_all()
    result = run_once(qc)      # '0' or '1'
    if result == "0":
        return "Heads"
    else:
        return "Tails"

for i in range(10):
    print(flip_coin())

In [ ]:
check_mission_1(flip_coin)

## Mission 2: An 8-sided die

Three qubits give 8 possible results, `000` through `111`. Read as binary numbers, those are 0 through 7. Add 1 and you have an 8-sided die.

Python turns a binary string into a normal number with `int("101", 2)`, which gives 5.

**TODO:** Put all three qubits (0, 1, and 2) into superposition.

In [ ]:
def make_d8_circuit():
    qc = QuantumCircuit(3)
    # TODO: put ALL THREE qubits into superposition

    qc.measure_all()
    return qc

def roll_d8():
    bits = run_once(make_d8_circuit())   # e.g. '101'
    return int(bits, 2) + 1              # '101' -> 5 -> 6

print([roll_d8() for i in range(10)])
make_d8_circuit().draw("mpl")

In [ ]:
check_mission_2(make_d8_circuit())

Roll it 1,000 times and look at the spread. A fair die gives bars of about the same height.

In [ ]:
rolls = [int(b, 2) + 1 for b in run_many(make_d8_circuit(), 1000)]
plt.hist(rolls, bins=range(1, 10), align="left", rwidth=0.8)
plt.xlabel("Roll")
plt.ylabel("How often")
plt.title("1,000 quantum d8 rolls")
plt.show()

## Mission 3: A fair 6-sided die

Six sides is trickier, because 3 qubits give 8 results, not 6. A shortcut like `number % 6` makes 1 and 2 come up more often than the rest, so the die would be unfair.

The fix is called **rejection sampling**: if you roll a number you can't use (6 or 7), throw it away and roll again.

**TODO:** Inside the loop, return `number + 1` only when `number` is less than 6.

In [ ]:
def roll_d6():
    while True:
        number = int(run_once(make_d8_circuit()), 2)   # 0 to 7
        # TODO: if number is less than 6, return number + 1. Otherwise the loop rolls again.
        return number + 1   # <- unfair: this can return 7 or 8!

print([roll_d6() for i in range(20)])

In [ ]:
check_mission_3(roll_d6)

## Mission 4: A loaded coin

The H gate always gives exactly 50/50. To set any odds you want, **tilt** the qubit with `qc.ry(theta, 0)`, where `theta` is an angle in radians. The chance of measuring 1 is

$$P(1) = \sin^2(\theta/2)$$

Some examples: `theta = 0` always gives 0, `theta = math.pi` always gives 1, and `theta = math.pi / 2` gives 50/50, just like H.

**TODO:** Make a coin that lands Heads (`0`) 75% of the time, so $P(1) = 0.25$. Hint: you need $\sin(\theta/2) = \sqrt{0.25}$, and `math.asin` undoes `sin`.

In [ ]:
theta = 0   # TODO: replace 0 with the right angle

qc = QuantumCircuit(1)
qc.ry(theta, 0)
qc.measure_all()

print("theta =", round(theta, 4))
print("Odds:", probabilities(qc))
plot_bloch_multivector(Statevector(qc.remove_final_measurements(inplace=False)))

In [ ]:
check_mission_4(qc)

The Bloch sphere shows what happened: the arrow tilted part of the way from 0 (up) toward 1 (down). The farther it tilts, the more likely you measure 1.

## Mission 5: Quantum fortune teller

Combine your 8-sided die with a list of 8 answers. Feel free to rewrite the answers.

**TODO:** Roll a quantum number from 0 to 7, then return the answer at that position in the list.

In [ ]:
answers = [
    "The qubits say yes.",
    "Definitely not.",
    "Ask again after lunch.",
    "Yes, but only in this universe.",
    "Your future is still in superposition.",
    "All signs point to yes.",
    "Unlikely. Try measuring in a different basis.",
    "Absolutely. Entangle with confidence.",
]

def fortune_teller(question):
    # TODO: get a quantum number from 0 to 7 (reuse make_d8_circuit), then return answers[number]
    return "TODO: pick an answer using a quantum number"

print(fortune_teller("Will our team win the hackathon?"))

In [ ]:
check_mission_5(fortune_teller)

## Make it yours

You have working quantum randomness. Now build something with it. Pick an idea below or invent your own:

- **Tabletop dice set.** Write one `roll(sides)` function that works for a d4, d6, d10, d12, or d20. How many qubits does a d20 need?
- **Quantum password generator.** Turn quantum bits into letters, numbers, and symbols. How many qubits do you need per character?
- **"Where should we eat?" picker** for spots around campus, with a loaded coin that favors your favorite.
- **Quantum vs. Python showdown.** Roll 10,000 times with each and compare histograms. Can you tell them apart?
- **Legendary fortunes.** Use `ry` so some fortunes are rare, like a loot drop in a video game.
- **A full dice game** for two players, with scoring and turns.

## Explain it

Judges will ask about the quantum part. Answer these in your own words, and be ready to say them out loud.

1. What does the H gate do to a qubit?
2. Why is quantum randomness different from Python's `random`?
3. How did you choose the angle for your loaded coin?

Our answers: